# Day 16 - Train in the notebook, save the artifact to S3

Run on the SageMaker notebook instance, kernel `conda_python3`.

The point today is the **plumbing**, not the model: data from S3 -> model in memory -> `model.tar.gz` + `metrics.json` back in S3,
in the layout SageMaker expects. Tomorrow a managed training job produces the same two files without this notebook doing the work.


In [ ]:
# A stop/start wipes pip installs - rerun this, then Kernel -> Restart Kernel.
# conda_python3 on this image ships without pyarrow or scikit-learn.
%pip install -q pyarrow scikit-learn joblib

In [ ]:
import json, tarfile, time
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import sklearn
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BUCKET = "dave-ds-lab-ca"
FEATURES_URI = f"s3://{BUCKET}/features/v1/"
MODEL_PREFIX = "models/manual/"
SEED = 42

FEATURES = [
    "passenger_count", "trip_distance", "trip_minutes", "fare_amount", "tolls_amount",
    "avg_mph", "fare_per_minute", "pickup_hour", "pickup_dow", "is_weekend", "is_rush_hour",
    "zone_tip_rate", "zone_trips", "zone_rolling_fare_20", "zone_prev_tip", "zone_seconds_since_prev",
]
TARGET = "tip_amount"
# Deliberately NOT features: pulocationid / dolocationid (raw IDs - zone_tip_rate is the encoded version),
# tpep_pickup_datetime (audit only), total_amount (contains the tip - Day 11).
print("sklearn", sklearn.__version__)

## 1. Load a sample, not everything

`ml.t3.medium` has 4 GB. We sample **inside Arrow** before converting to pandas, so the full 4.6M-row
training split never becomes a pandas DataFrame. `float32` halves memory again. 1.5M training rows is
plenty for 16 features; more data would move the third decimal of the metric, not the lesson.


In [ ]:
dataset = ds.dataset(FEATURES_URI, format="parquet", partitioning="hive")
rng = np.random.default_rng(SEED)

def load(split, n):
    t0 = time.perf_counter()
    table = dataset.to_table(columns=FEATURES + [TARGET], filter=ds.field("split") == split)
    idx = np.sort(rng.choice(table.num_rows, size=min(n, table.num_rows), replace=False))
    df = table.take(idx).to_pandas().astype("float32")
    del table
    print(f"{split:5s} {len(df):>10,} rows  {df.memory_usage().sum()/1e6:6.0f} MB  {time.perf_counter()-t0:5.1f}s")
    return df

train = load("train", 1_500_000)
valid = load("valid",   500_000)
X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]

## 2. Baselines first

A model number means nothing alone. Two baselines: predict the training mean for everyone,
and a one-line rule - *this zone's usual tip rate x the fare*. The model has to beat both to be worth anything.


In [ ]:
def score(y, pred):
    return {
        "mae":  round(float(mean_absolute_error(y, pred)), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y, pred))), 4),
        "r2":   round(float(r2_score(y, pred)), 4),
    }

results = {}
results["baseline_mean"] = score(y_valid, np.full(len(y_valid), y_train.mean()))
overall_rate = float((y_train / X_train["fare_amount"]).mean())
rule = X_valid["zone_tip_rate"].fillna(overall_rate) * X_valid["fare_amount"]
results["baseline_zone_rate_x_fare"] = score(y_valid, rule)
pd.DataFrame(results).T

## 3. Train

`HistGradientBoostingRegressor` is scikit-learn's LightGBM-style booster: fast on millions of rows,
and it handles the NaNs in `zone_prev_tip` / `zone_rolling_fare_20` natively (the first trip in a zone has no history).
Early stopping holds out 10% of the *training* rows - the March validation set stays untouched until scoring.


In [ ]:
params = dict(
    max_iter=300, learning_rate=0.1, max_leaf_nodes=63, min_samples_leaf=200,
    l2_regularization=1.0, early_stopping=True, validation_fraction=0.1,
    n_iter_no_change=15, random_state=SEED,
)
model = HistGradientBoostingRegressor(**params)

t0 = time.perf_counter()
model.fit(X_train, y_train)
train_seconds = time.perf_counter() - t0
print(f"trained in {train_seconds:.0f}s, {model.n_iter_} boosting rounds")

results["hgb"] = score(y_valid, model.predict(X_valid))
pd.DataFrame(results).T

## 4. Which features mattered

Permutation importance on a 50k validation slice: shuffle one column, see how much MAE gets worse.
If a single feature dominates suspiciously, suspect leakage before celebrating.


In [ ]:
from sklearn.inspection import permutation_importance

sl = slice(0, 50_000)
pi = permutation_importance(model, X_valid.iloc[sl], y_valid.iloc[sl],
                            scoring="neg_mean_absolute_error", n_repeats=3, random_state=SEED)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
imp.round(4)

## 5. Package the SageMaker way and upload

SageMaker's convention: a gzipped tar called `model.tar.gz` with the model file at the root.
Every hosting and batch-transform container looks for exactly that. Next to it, `metrics.json` -
every artifact carries its own scorecard, so nobody has to rerun a notebook to know if it was any good.


In [ ]:
out = Path("model_out"); out.mkdir(exist_ok=True)
joblib.dump(model, out / "model.joblib")

with tarfile.open(out / "model.tar.gz", "w:gz") as tar:
    tar.add(out / "model.joblib", arcname="model.joblib")

metrics = {
    "model": "sklearn.HistGradientBoostingRegressor",
    "sklearn_version": sklearn.__version__,
    "trained_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "data": {"features_uri": FEATURES_URI, "train_rows": len(X_train), "valid_rows": len(X_valid),
             "train_split": "2024-01..02", "valid_split": "2024-03"},
    "features": FEATURES, "target": TARGET, "params": params,
    "boosting_rounds": int(model.n_iter_), "train_seconds": round(train_seconds, 1),
    "valid": results,
    "top_features": imp.head(5).round(4).to_dict(),
}
(out / "metrics.json").write_text(json.dumps(metrics, indent=2))

s3 = boto3.client("s3")
for name in ["model.tar.gz", "metrics.json"]:
    s3.upload_file(str(out / name), BUCKET, MODEL_PREFIX + name)
    size = s3.head_object(Bucket=BUCKET, Key=MODEL_PREFIX + name)["ContentLength"]
    print(f"s3://{BUCKET}/{MODEL_PREFIX}{name}  {size/1e6:.2f} MB")

## 6. Prove the artifact round-trips

Download it back, unpack into a fresh folder, predict. If this cell works, a hosting container will too.


In [ ]:
check = Path("roundtrip"); check.mkdir(exist_ok=True)
s3.download_file(BUCKET, MODEL_PREFIX + "model.tar.gz", str(check / "model.tar.gz"))
with tarfile.open(check / "model.tar.gz") as tar:
    tar.extractall(check)
reloaded = joblib.load(check / "model.joblib")

same = np.allclose(reloaded.predict(X_valid.head(1000)), model.predict(X_valid.head(1000)))
print("reloaded model gives identical predictions:", same)
print(json.dumps(metrics["valid"], indent=2))

## 7. STOP THE INSTANCE

SageMaker console -> Notebooks -> `ds-lab-notebook` -> Actions -> **Stop**.
